In [1]:
from embedder import Embedder

embed = Embedder()

2026-07-13 21:19:44.804127711 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
uv add openai pydantic python-dotenv pandas

/workspaces/llm-zoomcamp-2026-code/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


Module 4 Homework starts...

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [5]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

Building the ground truth dataset...

In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
from evaluation_utils import llm_structured_retry

In [9]:
import json

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [10]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [11]:
len(documents)

72

In [12]:
documents[1]

{'content': '# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nFor this module, all you need is Python with Jupyter.\n\n## Prerequisites\n\nYou need the following:\n\n- Python (3.14 or later)\n- An [OpenAI account](https://openai.com/) (or an OpenAI-compatible\n  provider like Groq, Gemini, or Ollama)\n- Basic familiarity with Python and the command line\n\n## Creating the project\n\nWe\'ll start from scratch - no cloning needed. You\'ll create the\nproject yourself, step by step.\n\nFirst, install uv. It\'s a Python package manager, and I switched all my\nprojects to it because it\'s fast and convenient. Once I started using\nit, I never wanted to go back.\n\nOn Mac or Linux:\n\n```bash\ncurl -LsSf https://astral.sh/uv/install.sh | sh\n```\n\nOn Windows:\n\n```powershell\npowershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"\n```\n\n(You can also use `pip install uv` if you p

In [13]:
len(ground_truth)

15

In [14]:
ground_truth[8]

{'question': 'What’s the safest way to keep my API key out of git, and what should go in the .env file?',
 'document': '01-agentic-rag/lessons/02-environment.md'}

In [15]:
len(usages)

3

In [16]:
usages[2]

ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=99, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1852)

In [17]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [18]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/72 [00:00<?, ?it/s]

In [19]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

360

360=72*5, 5 q'uestions generated from answer. So correct

In [20]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.11398949999999998

In [21]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.11398949999999998

In [22]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [23]:
df_ground_truth.to_csv("data/ground_truth-homeowrk.csv", index=False)

In [24]:
df_ground_truth.head()

,question,document
0,What is retrieval-augmented generation trying ...,01-agentic-rag/lessons/01-intro.md
1,Why does this lesson treat an LLM like a black...,01-agentic-rag/lessons/01-intro.md
2,What are the main limits of LLMs that make RAG...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in this module to m...,01-agentic-rag/lessons/01-intro.md
4,How is Part 2 different from Part 1 in this RA...,01-agentic-rag/lessons/01-intro.md


Question 2:

In [25]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [26]:
len(chunks)

295

In [27]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [28]:
# A. Building Text Index
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [29]:
def text_search(query):
    #boost_dict = {"question": 3.0, "section": 0.5}    

    return index.search(
        query,
        num_results=5,
        #boost_dict=boost_dict
    )

In [30]:
#B. Building Vector Index

# embed every chunk's content with encode_batch
batch_vectors=embed.encode_batch([chunk['content'] for chunk in chunks])
len(batch_vectors)


295

In [31]:
import numpy as np
X = np.array(batch_vectors)

In [32]:
from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks) 

In [33]:
def vector_search(query):
    # 1. Transform the text query into an embedding vector
    query_vector = embed.encode(query)
    
    # 2. Search using the query vector
    results = vindex.search(
        query_vector,
        num_results=5
    )
    return results

In [34]:
q = ground_truth[0]["question"]
q

'What is retrieval-augmented generation trying to fix with language models?'

In [35]:
text_search(q)

[{'start': 2000,
  'content': 'you want to receive a certificate, you need to submit your project while we\'re still accepting submissions.\n\nCourse: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nYou don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nWhat is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?\nThe zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.\n\nCloud alternatives with GPU\nCheck the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.\n"""\n```\n\nNotice the prompt doesn\'t end with `Answer:`. With older models like\nGPT-3 we added that to nudge the model into completing the

In [36]:
vector_search(q)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

Question 4: Evaluate Text Search Hit Rate:

In [37]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [38]:
ground_truth[0]

{'question': 'What is retrieval-augmented generation trying to fix with language models?',
 'document': '01-agentic-rag/lessons/01-intro.md'}

In [39]:
compute_relevance_text(ground_truth[0])

[0, 0, 0, 1, 0]

In [40]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [41]:
relevance_total_text = compute_relevance_total_text(ground_truth)

  0%|          | 0/360 [00:00<?, ?it/s]

In [42]:
len(relevance_total_text)

360

In [43]:
relevance_total_text[10]

[1, 0, 0, 0, 0]

In [44]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [45]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [46]:
relevance_total = compute_relevance_total(ground_truth, text_search)
relevance_total

  0%|          | 0/360 [00:00<?, ?it/s]

[[0, 0, 0, 1, 0],
 [1, 1, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 1],
 [1, 1, 1, 1, 1],
 [0, 0, 0, 1, 0],
 [1, 1, 1, 1, 1],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 0, 1, 0, 1],
 [1, 1, 0, 0, 1],
 [0, 1, 1, 0, 1],
 [1, 1, 1, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 1, 0],
 [1, 1, 0, 1, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0,

In [47]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [48]:
hit_rate(relevance_total)

0.7833333333333333

Question 5: Evaluating Vector Search MMR

In [49]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [50]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [51]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7833333333333333, 'mrr': 0.6072685185185185}

In [52]:
evaluate(
    ground_truth,
    vector_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.75, 'mrr': 0.5671296296296297}

Question 6: Tuning Hybrid Search

In [53]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [54]:
def text_search(query,num_results):
    #boost_dict = {"question": 3.0, "section": 0.5}    

    return index.search(
        query,
        num_results=num_results,
        #boost_dict=boost_dict
    )

In [55]:
def vector_search(query,num_results):
    # 1. Transform the text query into an embedding vector
    query_vector = embed.encode(query)
    
    # 2. Search using the query vector
    results = vindex.search(
        query_vector,
        num_results=num_results
    )
    return results

In [62]:
def hybrid_search(query, k=200):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([vector_results,text_results], k=k)

In [63]:
evaluate(
    ground_truth,
    hybrid_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8472222222222222, 'mrr': 0.6391666666666668}